<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/gensim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gensim Basics — Full Notes + Installation + Exercises

Easy explanation + code. Covers installation, core building blocks, Word2Vec (train your own + use pretrained), TF-IDF, and Topic Modeling (LDA).

## 1. What is Gensim? (Easy Explanation)

You've already covered **spaCy** and **NLTK** — both are general-purpose NLP toolkits (tokenizing, POS tagging, NER, lemmatization, etc.).

**Gensim is different — it specializes in just two things:**
1. **Training and using word embeddings** (Word2Vec, FastText, Doc2Vec) — turning words/documents into meaningful vectors, similar to what you did with spaCy's pre-trained vectors, except Gensim lets you **train your own** embeddings from your own text.
2. **Topic Modeling** — automatically discovering the "hidden topics" inside a large collection of documents (e.g. "this batch of news articles secretly clusters into sports, politics, and tech topics") — something neither spaCy nor NLTK does out of the box.

```
spaCy / NLTK  -> general text processing (tokenize, POS tag, NER, lemmatize...)
Gensim        -> specialized: train embeddings + discover topics in large document collections
```

**When would you actually reach for Gensim instead of spaCy's pre-trained vectors?**
- You have a LOT of domain-specific text (e.g. medical records, legal documents) and want embeddings trained specifically on YOUR vocabulary, not generic Wikipedia text.
- You want to explore what topics exist in a large, unlabeled set of documents (topic modeling) — this is unsupervised, you don't even need labels.

## 2. Installation

Install via pip:
```
pip install gensim
```

If you're on Google Colab, gensim is usually pre-installed, but you can force the latest version with:
```
!pip install --upgrade gensim
```

**Common installation issues:**
- Gensim depends on `numpy` and `scipy` — if you hit build errors, first upgrade those: `pip install --upgrade numpy scipy`
- On some systems, Gensim also benefits from a fast BLAS backend for speed (not required to just get started, only matters for training very large models).
- Always check the installed version, since the API has changed across major versions (4.x removed some things that existed in 3.x, like the old `.wv.vocab` attribute):

In [2]:
#!pip install --upgrade gensim

In [3]:
import gensim

print(gensim.__version__)   # good practice: always know your gensim version since the API changed across major releases


4.4.0


---
## 3. Core Building Block 1 — Tokenizing with `simple_preprocess`

Gensim has its own lightweight tokenizer/cleaner — lowercases everything, removes punctuation, and drops very short tokens automatically.

In [4]:
import gensim.utils

text = "Hello World! Gensim is GREAT for NLP tasks... isn't it?"

tokens = gensim.utils.simple_preprocess(text)
print(tokens)


['hello', 'world', 'gensim', 'is', 'great', 'for', 'nlp', 'tasks', 'isn', 'it']


**Output:** `['hello', 'world', 'gensim', 'is', 'great', 'for', 'nlp', 'tasks', 'isn', 'it']`

Notice: lowercased, punctuation gone, and this is much simpler/rougher than spaCy's tokenizer (no POS tags, no lemmatization) — Gensim isn't trying to compete with spaCy here, it just needs SOME quick tokenizer to feed into its models.

## 4. Core Building Block 2 — Dictionary (Word <-> ID mapping)

Before Gensim can do math on your text, every unique word needs a numeric ID. A `Dictionary` builds this mapping automatically from a collection of tokenized documents.

In [5]:
from gensim.corpora import Dictionary

documents = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are great pets"
]

tokenized_docs = [gensim.utils.simple_preprocess(doc) for doc in documents]
print(tokenized_docs)


[['the', 'cat', 'sat', 'on', 'the', 'mat'], ['the', 'dog', 'sat', 'on', 'the', 'log'], ['cats', 'and', 'dogs', 'are', 'great', 'pets']]


In [6]:
dictionary = Dictionary(tokenized_docs)

print(dictionary.token2id)   # word -> unique integer ID mapping


{'cat': 0, 'mat': 1, 'on': 2, 'sat': 3, 'the': 4, 'dog': 5, 'log': 6, 'and': 7, 'are': 8, 'cats': 9, 'dogs': 10, 'great': 11, 'pets': 12}


In [7]:
print(len(dictionary))   # total number of unique words (vocabulary size)


13


## 5. Core Building Block 3 — Bag of Words Corpus (`doc2bow`)

Once you have a `Dictionary`, you can convert any tokenized document into a Bag-of-Words representation: a list of `(word_id, count)` pairs.

In [8]:
bow_corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

for doc_bow in bow_corpus:
    print(doc_bow)


[(0, 1), (1, 1), (2, 1), (3, 1), (4, 2)]
[(2, 1), (3, 1), (4, 2), (5, 1), (6, 1)]
[(7, 1), (8, 1), (9, 1), (10, 1), (11, 1), (12, 1)]


In [9]:
# translate the numeric IDs back into readable words, to double check what's happening
for doc_bow in bow_corpus:
    print([(dictionary[word_id], count) for word_id, count in doc_bow])


[('cat', 1), ('mat', 1), ('on', 1), ('sat', 1), ('the', 2)]
[('on', 1), ('sat', 1), ('the', 2), ('dog', 1), ('log', 1)]
[('and', 1), ('are', 1), ('cats', 1), ('dogs', 1), ('great', 1), ('pets', 1)]


## 6. TF-IDF in Gensim

Just like sklearn's `TfidfVectorizer` (covered earlier), Gensim has its own TF-IDF implementation that works on top of the Bag-of-Words corpus you just built.

In [10]:
from gensim.models import TfidfModel

tfidf = TfidfModel(bow_corpus)

for doc_bow in bow_corpus:
    doc_tfidf = tfidf[doc_bow]   # transform this document's BOW into TF-IDF weighted scores
    print([(dictionary[word_id], round(score, 3)) for word_id, score in doc_tfidf])


[('cat', np.float64(0.596)), ('mat', np.float64(0.596)), ('on', np.float64(0.22)), ('sat', np.float64(0.22)), ('the', np.float64(0.44))]
[('on', np.float64(0.22)), ('sat', np.float64(0.22)), ('the', np.float64(0.44)), ('dog', np.float64(0.596)), ('log', np.float64(0.596))]
[('and', np.float64(0.408)), ('are', np.float64(0.408)), ('cats', np.float64(0.408)), ('dogs', np.float64(0.408)), ('great', np.float64(0.408)), ('pets', np.float64(0.408))]


Notice "the" (which appears in almost every document) gets a LOW or missing TF-IDF weight, while rarer words like "log", "pets" get higher weights — same idea as sklearn's TF-IDF, just computed via Gensim's pipeline.

---
## 7. Word2Vec — Training Your OWN Word Embeddings

Recall from the spaCy word-vectors notebook: `en_core_web_lg` ships PRE-TRAINED vectors (trained on Wikipedia by someone else). With Gensim's `Word2Vec`, you can train your OWN embeddings from scratch, on YOUR OWN text — useful when your vocabulary is specialized (medical, legal, gaming slang, etc.) and generic pre-trained vectors won't capture it well.

In [11]:
from gensim.models import Word2Vec

# a slightly bigger toy corpus so the model has more to learn from
sentences = [
    "the king ruled the kingdom wisely".split(),
    "the queen ruled the kingdom wisely".split(),
    "the man walked to the market".split(),
    "the woman walked to the market".split(),
    "the king and queen visited the market".split(),
    "the dog ran across the park".split(),
    "the cat ran across the park".split(),
    "the dog and cat played in the park".split(),
]

# vector_size: how many numbers per word (like spaCy's 300, but you choose)
# window: how many neighboring words to look at for context
# min_count: ignore words that appear fewer than this many times
model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=4)


**Important — with such a tiny toy corpus, the resulting vectors won't be very meaningful.** Real Word2Vec models need thousands to millions of sentences to learn genuinely useful relationships. This toy example is just to show the MECHANICS of training; treat the actual similarity numbers below as illustrative, not meaningful.

In [12]:
model.wv["king"].shape   # 50-number vector, as we requested with vector_size=50


(50,)

In [13]:
model.wv.most_similar("king")   # words whose vectors ended up closest to "king", based on this tiny corpus


[('wisely', 0.5293555855751038),
 ('the', 0.2104354053735733),
 ('kingdom', 0.18857939541339874),
 ('walked', 0.1187339648604393),
 ('ran', 0.08190789818763733),
 ('man', 0.08008356392383575),
 ('ruled', 0.054336268454790115),
 ('market', 0.03234602138400078),
 ('park', 0.011446892283856869),
 ('woman', -0.001321150572039187)]

In [14]:
model.wv.similarity("king", "queen")   # cosine similarity between two words


np.float32(-0.07937872)

## 8. Using Pre-Trained Word2Vec Models (Gensim Downloader API)

Just like spaCy's `en_core_web_lg`, Gensim gives you access to several famous PRE-TRAINED embedding models (trained on huge real-world corpora), so you don't always have to train your own.

In [15]:
import gensim.downloader as api

# list all available pre-trained models/datasets (long list, this just previews the first several)
list(api.info()["models"].keys())[:10]


['fasttext-wiki-news-subwords-300',
 'conceptnet-numberbatch-17-06-300',
 'word2vec-ruscorpora-300',
 'word2vec-google-news-300',
 'glove-wiki-gigaword-50',
 'glove-wiki-gigaword-100',
 'glove-wiki-gigaword-200',
 'glove-wiki-gigaword-300',
 'glove-twitter-25',
 'glove-twitter-50']

In [16]:
# download a small, fast pre-trained model (~66MB) -- good for quick experimentation
# NOTE: this requires an internet connection and can take a minute or two the first time
wv = api.load("glove-wiki-gigaword-50")


[==================================================] 100.0% 66.0/66.0MB downloaded


In [17]:
wv.most_similar("computer")


[('computers', 0.9165045022964478),
 ('software', 0.8814992904663086),
 ('technology', 0.852556049823761),
 ('electronic', 0.812586784362793),
 ('internet', 0.8060455322265625),
 ('computing', 0.802603542804718),
 ('devices', 0.8016185760498047),
 ('digital', 0.7991793751716614),
 ('applications', 0.7912740707397461),
 ('pc', 0.7883159518241882)]

In [18]:
wv.similarity("king", "queen")


np.float32(0.7839043)

**Bigger, more accurate pre-trained options** (much larger downloads, only run if you have time/bandwidth):
```python
wv_big = api.load("word2vec-google-news-300")   # ~1.6 GB, Google's famous News-trained vectors
```

### The Famous Analogy Test — Now in Gensim

Same idea as the spaCy word-vectors notebook (king - man + woman ≈ queen), but Gensim has a convenient built-in function for it: `most_similar(positive=[...], negative=[...])`.

In [19]:
wv.most_similar(positive=["king", "woman"], negative=["man"], topn=5)


[('queen', 0.8523604273796082),
 ('throne', 0.7664334177970886),
 ('prince', 0.7592144012451172),
 ('daughter', 0.7473883628845215),
 ('elizabeth', 0.7460219860076904)]

**Output:** "queen" should appear at or near the top of this list — same analogy result as before, just computed with Gensim's convenience function instead of manual vector math.

---
## 9. Saving and Loading a Trained Model

Training word vectors can take a while, so you'll usually want to save your model and reload it later instead of retraining every time.

In [20]:
model.save("my_word2vec.model")


In [21]:
loaded_model = Word2Vec.load("my_word2vec.model")
loaded_model.wv.most_similar("king")


[('wisely', 0.5293555855751038),
 ('the', 0.2104354053735733),
 ('kingdom', 0.18857939541339874),
 ('walked', 0.1187339648604393),
 ('ran', 0.08190789818763733),
 ('man', 0.08008356392383575),
 ('ruled', 0.054336268454790115),
 ('market', 0.03234602138400078),
 ('park', 0.011446892283856869),
 ('woman', -0.001321150572039187)]

For pre-trained `KeyedVectors` objects (like the `wv` we downloaded via `api.load`), use `save`/`load` from `KeyedVectors` instead:
```python
wv.save("glove_vectors.kv")
from gensim.models import KeyedVectors
loaded_wv = KeyedVectors.load("glove_vectors.kv")
```

---
## 10. Topic Modeling with LDA (Latent Dirichlet Allocation)

This is Gensim's other big specialty: given a pile of documents with NO labels, automatically discover what "topics" they cluster into. Each topic is represented as a group of words that tend to appear together.

In [22]:
documents = [
    "the stock market rallied as tech shares surged today",
    "investors reacted to the central bank interest rate decision",
    "the football team won the championship after a great season",
    "the striker scored twice in the final match of the tournament",
    "researchers discovered a new exoplanet using a powerful telescope",
    "scientists published a study on renewable energy storage technology"
]

tokenized_docs = [gensim.utils.simple_preprocess(doc) for doc in documents]

dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]


In [23]:
from gensim.models import LdaModel

# num_topics: how many hidden topics to look for (you choose this, like choosing K in K-means)
lda_model = LdaModel(corpus, num_topics=3, id2word=dictionary, passes=10, random_state=42)


In [24]:
for idx, topic in lda_model.print_topics(num_words=5):
    print(f"Topic {idx}: {topic}")


Topic 0: 0.099*"the" + 0.031*"striker" + 0.031*"final" + 0.031*"scored" + 0.031*"in"
Topic 1: 0.095*"the" + 0.054*"team" + 0.054*"season" + 0.054*"won" + 0.054*"after"
Topic 2: 0.042*"powerful" + 0.042*"researchers" + 0.042*"discovered" + 0.042*"exoplanet" + 0.042*"telescope"


In [25]:
# check which topic(s) a NEW, unseen document belongs to
new_doc = "the company announced record profits this quarter"
new_bow = dictionary.doc2bow(gensim.utils.simple_preprocess(new_doc))

lda_model.get_document_topics(new_bow)


[(0, np.float32(0.6106663)),
 (1, np.float32(0.21347454)),
 (2, np.float32(0.17585921))]

**Output:** a list of `(topic_id, probability)` pairs — showing how strongly this new document belongs to each discovered topic. With business-related words like "company" and "profits", it should lean heavily toward the BUSINESS-like topic.

---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Check version | `gensim.__version__` |
| Simple tokenizer | `gensim.utils.simple_preprocess(text)` |
| Build word<->id dictionary | `Dictionary(tokenized_docs)` |
| Bag of Words for one doc | `dictionary.doc2bow(tokens)` |
| TF-IDF | `TfidfModel(bow_corpus)` then `tfidf[doc_bow]` |
| Train Word2Vec | `Word2Vec(sentences, vector_size=100, window=5, min_count=1)` |
| Get a word's vector | `model.wv["word"]` |
| Similar words | `model.wv.most_similar("word")` |
| Word similarity | `model.wv.similarity("w1", "w2")` |
| Analogy (a - b + c) | `wv.most_similar(positive=["a","c"], negative=["b"])` |
| Load pre-trained vectors | `gensim.downloader.load("glove-wiki-gigaword-50")` |
| Save / load model | `model.save(path)` / `Word2Vec.load(path)` |
| Topic modeling | `LdaModel(corpus, num_topics=K, id2word=dictionary)` |
| Topics for a new doc | `lda_model.get_document_topics(new_bow)` |

## Gensim vs spaCy vs NLTK — Quick Reference

| | NLTK | spaCy | Gensim |
|---|---|---|---|
| Tokenizing, POS, NER | Yes (older style) | Yes (modern, fast) | Very basic only |
| Pre-trained word vectors | No (not built-in) | Yes (`en_core_web_lg`) | Yes (via downloader API) |
| TRAIN your own word vectors | No | No | **Yes** (its specialty) |
| Topic modeling (LDA etc.) | No | No | **Yes** (its specialty) |

---

---
# Practice Exercises (with Solutions)

## Exercise 1 — Build a Dictionary + BOW Corpus from a New Text Collection

**Task:** Given the sentences below, build a Gensim `Dictionary` and Bag-of-Words corpus, then print each document in readable (word, count) form.

In [26]:
docs = [
    "spacy and nltk are great libraries for nlp",
    "gensim is great for topic modeling and word embeddings",
    "nlp libraries help process and understand text"
]

tokenized = [gensim.utils.simple_preprocess(d) for d in docs]
dictionary_ex1 = Dictionary(tokenized)
bow_ex1 = [dictionary_ex1.doc2bow(d) for d in tokenized]

for doc_bow in bow_ex1:
    print([(dictionary_ex1[wid], count) for wid, count in doc_bow])


[('and', 1), ('are', 1), ('for', 1), ('great', 1), ('libraries', 1), ('nlp', 1), ('nltk', 1), ('spacy', 1)]
[('and', 1), ('for', 1), ('great', 1), ('embeddings', 1), ('gensim', 1), ('is', 1), ('modeling', 1), ('topic', 1), ('word', 1)]
[('and', 1), ('libraries', 1), ('nlp', 1), ('help', 1), ('process', 1), ('text', 1), ('understand', 1)]


## Exercise 2 — Train a Small Word2Vec Model and Find Similar Words

**Task:** Using the same 3 sentences above, train a tiny Word2Vec model and find words most similar to "nlp".

In [27]:
sentences_ex2 = tokenized   # reuse tokenized docs from Exercise 1

model_ex2 = Word2Vec(sentences_ex2, vector_size=20, window=2, min_count=1, workers=1)

model_ex2.wv.most_similar("nlp")


[('word', 0.25416165590286255),
 ('are', 0.1588306427001953),
 ('libraries', 0.15222927927970886),
 ('spacy', 0.14858750998973846),
 ('is', 0.12820808589458466),
 ('embeddings', 0.07645351439714432),
 ('topic', 0.07290135324001312),
 ('text', 0.06166551262140274),
 ('and', 0.05017896369099617),
 ('gensim', 0.04806629940867424)]

**Note:** with only 3 short training sentences, results are just illustrative of the mechanics — real Word2Vec needs far more data to produce meaningful similarities.

## Exercise 3 — Run LDA Topic Modeling and Interpret the Topics

**Task:** Using the same 3 sentences, run LDA with 2 topics and print the top words for each.

In [28]:
corpus_ex3 = bow_ex1   # reuse the BOW corpus from Exercise 1

lda_ex3 = LdaModel(corpus_ex3, num_topics=2, id2word=dictionary_ex1, passes=10, random_state=42)

for idx, topic in lda_ex3.print_topics(num_words=5):
    print(f"Topic {idx}: {topic}")


Topic 0: 0.104*"and" + 0.104*"libraries" + 0.104*"nlp" + 0.062*"are" + 0.062*"spacy"
Topic 1: 0.084*"for" + 0.084*"great" + 0.083*"and" + 0.083*"gensim" + 0.083*"modeling"


**Expected pattern:** with such a small, overlapping corpus ("nlp", "great", "libraries" repeat across sentences), the 2 topics may not separate very cleanly — this is expected. LDA needs a reasonably large, more DIVERSE document collection to find clean, distinct topics; 3 short overlapping sentences is really too small a dataset to demonstrate LDA's full power, but it shows the mechanics correctly.